# Chapter 13 — DPO: Direct Preference Optimization
**Math for ML Engineers** · companion notebook

This notebook mirrors §13.8 (DPO) on a toy discrete preference task where a policy over 4 prompts × 3 completions is trained to prefer human-annotated completions.

**Equations:** DPO loss (13.46), implicit reward (13.44), KL divergence monitor (13.40).

No GPU needed — pure NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Problem setup

We have $|\mathcal{X}| = 4$ prompts and $|\mathcal{Y}| = 3$ completions per prompt.  The policy $\pi_\theta(y \mid x) = \text{softmax}(\theta_x)_y$ is a categorical distribution parameterised by a $4 \times 3$ logit table.  The reference policy is $\pi_{\text{ref}} = \text{uniform}$ (equal logits).

**Preference pairs:** for each prompt $x$, one completion $y_w$ is marked preferred and one $y_l$ is marked rejected.

In [ ]:
N_PROMPTS, N_COMPL = 4, 3
rng = np.random.default_rng(42)

# Preference pairs: (prompt, chosen, rejected)
pairs = [(0, 0, 2), (1, 1, 0), (2, 2, 1), (3, 0, 1)]

def softmax_policy(theta, x):
    '''Returns probability vector over completions for prompt x.'''
    logits = theta[x]
    logits = logits - logits.max()
    e = np.exp(logits)
    return e / e.sum()

# Reference policy: uniform
ref_logp = np.zeros((N_PROMPTS, N_COMPL)) - np.log(N_COMPL)
print('Reference policy (uniform):')
theta0 = np.zeros((N_PROMPTS, N_COMPL))
print(np.exp(ref_logp))


## DPO loss

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y_w,y_l)} \left[\log \sigma\!\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

The quantity inside $\sigma(\cdot)$ is the **reward margin**: how much more the policy prefers $y_w$ over $y_l$ \_relative to the reference\_. A positive margin means the policy has learned the preference.

Critical: $\pi_{\text{ref}}$ must be **frozen** (see Failure Mode 11).

In [ ]:
def dpo_loss_and_grad(theta, ref_logp, pairs, beta=0.5):
    total_loss = 0.0
    grad = np.zeros_like(theta)
    for (x, yw, yl) in pairs:
        p = softmax_policy(theta, x)           # current policy probs
        log_pol_w = np.log(p[yw] + 1e-10)
        log_pol_l = np.log(p[yl] + 1e-10)
        log_ref_w = ref_logp[x, yw]
        log_ref_l = ref_logp[x, yl]
        # reward margin h
        h = beta * ((log_pol_w - log_ref_w) - (log_pol_l - log_ref_l))
        sig = 1.0 / (1.0 + np.exp(-h))
        total_loss += -np.log(sig + 1e-10)
        # gradient: chain rule through sigmoid and log-softmax
        coeff = -(1 - sig) * beta
        for y, sign in [(yw, +1.0), (yl, -1.0)]:
            # d log pi(y|x) / d theta[x,k] = 1[y==k] - pi(k|x)
            dlogp = -p.copy()
            dlogp[y] += 1.0
            grad[x] += coeff * sign * dlogp
    return total_loss / len(pairs), grad / len(pairs)

# Verify gradient (finite difference)
t = np.random.default_rng(1).standard_normal((N_PROMPTS, N_COMPL)) * 0.1
loss0, g = dpo_loss_and_grad(t, ref_logp, pairs)
eps = 1e-5
fd = np.zeros_like(t)
for i in range(N_PROMPTS):
    for j in range(N_COMPL):
        t2 = t.copy(); t2[i,j] += eps
        fd[i,j] = (dpo_loss_and_grad(t2, ref_logp, pairs)[0] - loss0) / eps
print(f'Gradient max error (finite diff check): {np.abs(g - fd).max():.2e}')


## Training loop

We run gradient descent on $\mathcal{L}_{\text{DPO}}$ and monitor:
- **Loss** — should decrease
- **Reward margin** — should go positive for all pairs
- **KL divergence** $\text{KL}(\pi_\theta \| \pi_{\text{ref}})$ — controlled by $\beta$; divergence signals reward hacking (Failure Mode 7)

In [ ]:
def kl_div(theta, ref_logp):
    kl = 0.0
    for x in range(N_PROMPTS):
        p = softmax_policy(theta, x)
        kl += np.sum(p * (np.log(p + 1e-10) - ref_logp[x]))
    return kl / N_PROMPTS

theta = np.zeros((N_PROMPTS, N_COMPL))
losses, kls, margins = [], [], []

for step in range(400):
    loss, g = dpo_loss_and_grad(theta, ref_logp, pairs, beta=0.5)
    theta -= 0.1 * g
    kl = kl_div(theta, ref_logp)
    # reward margins for all pairs
    ms = []
    for (x, yw, yl) in pairs:
        p = softmax_policy(theta, x)
        m = np.log(p[yw]+1e-10) - np.log(p[yl]+1e-10)
        ms.append(m)
    losses.append(loss)
    kls.append(kl)
    margins.append(np.mean(ms))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(losses, 'C0'); axes[0].set_title('DPO Loss'); axes[0].set_xlabel('Step')
axes[1].plot(kls, 'C1'); axes[1].axhline(0, ls='--', color='gray')
axes[1].set_title('KL(π_θ ∥ π_ref)'); axes[1].set_xlabel('Step')
axes[2].plot(margins, 'C2'); axes[2].axhline(0, ls='--', color='gray')
axes[2].set_title('Mean reward margin (chosen − rejected log ratio)')
axes[2].set_xlabel('Step')
plt.tight_layout(); plt.show()


## β sensitivity: KL budget vs. alignment quality

The scalar $\beta$ controls the tradeoff: small $\beta$ allows the policy to diverge far from the reference (high KL, strong preference satisfaction); large $\beta$ keeps the policy close (low KL, weaker margins).  The production default is $\beta = 0.1$.

In [ ]:
results = {}
for beta in [0.05, 0.1, 0.5, 2.0]:
    theta_b = np.zeros((N_PROMPTS, N_COMPL))
    final_kl, final_margin = 0, 0
    for _ in range(400):
        _, g = dpo_loss_and_grad(theta_b, ref_logp, pairs, beta=beta)
        theta_b -= 0.1 * g
    final_kl = kl_div(theta_b, ref_logp)
    ms = []
    for (x, yw, yl) in pairs:
        p = softmax_policy(theta_b, x)
        ms.append(np.log(p[yw]+1e-10) - np.log(p[yl]+1e-10))
    final_margin = np.mean(ms)
    results[beta] = (final_kl, final_margin)
    print(f'β={beta:.2f}  KL={final_kl:.3f}  margin={final_margin:.3f}')

betas_list = sorted(results)
kls_list = [results[b][0] for b in betas_list]
marg_list = [results[b][1] for b in betas_list]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].semilogx(betas_list, kls_list, 'o-C1'); axes[0].set_xlabel('β'); axes[0].set_ylabel('KL(π_θ∥π_ref)')
axes[0].set_title('KL divergence from reference')
axes[1].semilogx(betas_list, marg_list, 'o-C2'); axes[1].set_xlabel('β'); axes[1].set_ylabel('reward margin')
axes[1].set_title('Mean preference margin')
plt.tight_layout(); plt.show()


## Exercise

1. Add 5% label noise: randomly flip 1 in 20 preference pairs.  How does DPO behave compared to clean labels?  The book states that $>20\%$ label noise degrades DPO (Appendix D checklist) — can you verify empirically?
2. Implement the 'frozen reference accidentally updated' failure (Failure Mode 11): after 200 steps, set `ref_logp = np.log(softmax_policy_table(theta))` and continue training.  What happens to the loss and KL? Does the model lose its alignment?